# 05 – Directly mapped types and final output

Reads the output of **04** (`nisjemodel-remote-sensing_norge_2026_25833.geo.parquet`)
and overrides the classification where directly mapped areas exist:

1. **NA-MC08 – Marint bunnsystem preget av oksygenmangel**
   from the MDir water type classification `NyTypologi2022`, types **1a, 2a, 3a**
   (`… med oksygenfattig bunnvann`, see `mnk.rules.WATER_TYPE_NAMES`).
   Only bottom with `nkm_lys` afotisk or eufotisk and `nkm_lm-tv = LM-TV_0` (permanently submerged, not intertidal).
2. **NA-MJ01 – Taretrålingsbunn**
   from NIVA kelp trawling areas 2020–2025. Applies to all bottom inside the areas.

The rules are defined in `mnk.rules.DIRECT_RULES`. Layers are applied in order; a later layer overrides an earlier one where they overlap.
Only affected polygons (spatial index) go through geometry operations.

The ungridded result is saved as `marint-naturkart-ungridded_norge_2026_25833.geo.parquet`
(EPSG:25833, source for PMTiles). Finally everything is split along the EEA 1 km grid
(`rute_fkid`) in EPSG:3035 and written as `marint-naturkart_norge_2026_3035`
(GeoParquet).

In [ ]:
import gc
import json
import tempfile
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import shapely
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

import mnk
import mnk.rules
import mnk.attributes
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

IN_PARQUET = REPO / f"{mnk.utils.to_filename('nisjemodel-remote-sensing', 'norge', '2026', '25833')}.geo.parquet"
# EEA reference grid (1 km), native EPSG:3035 LAEA. The grid split (§5) runs in 3035,
GRID_URL = "gs://niva-geodata/MarintNaturKart/aux/NO_1km.geo.parquet"

for rule in mnk.rules.DIRECT_RULES:
    print(rule["nin_kode_fkid"], mnk.rules.NIN_NAMES[rule["nin_kode_fkid"]], rule["url"])

NA-MC08 Marint bunnsystem preget av oksygenmangel gs://niva-geodata/MarintNaturKart/input/mdir/NyTypologi2022.geo.parquet
NA-MJ01 Taretrålingsbunn gs://niva-geodata/MarintNaturKart/input/niva/taretraling_polygons_2020-2025_2026-11-15.gpkg


## 1. Load data

In [2]:
final = gpd.read_parquet(IN_PARQUET)
assert final.crs.to_epsg() == 25833
print(f"Input (04): {len(final):,} polygons")


def load_mask(rule):
    """Read the direct dataset of *rule* and return clean single polygons."""
    url = rule["url"]
    gdf = gpd.read_parquet(url) if url.endswith(".parquet") else gpd.read_file(url)
    if rule["column"] is not None:
        gdf = gdf.loc[gdf[rule["column"]].isin(rule["values"])]
    gdf = gdf[["geometry"]].to_crs(CRS)
    gdf = gdf.set_geometry(shapely.make_valid(gdf.geometry.to_numpy()), crs=CRS)
    gdf = gdf.explode(index_parts=False)
    gdf = gdf[gdf.geom_type == "Polygon"].reset_index(drop=True)
    print(f"{rule['nin_kode_fkid']}: {len(gdf):,} polygons ({gdf.area.sum() / 1e6:,.1f} km²)")
    return gdf

Input (04): 4,977,343 polygons


## 2. Generic override

`override` splits affected polygons into the part inside the mask (new type) and
the remainder (unchanged), same pattern as in 04.

In [3]:
def override(final, mask, code, method, source, candidates=None):
    """Assign NiN type *code* to the parts of *final* inside *mask*."""
    if candidates is None:
        candidates = pd.Series(True, index=final.index)
    sub = final.loc[candidates]
    hit = sub.sindex.query(mask.geometry, predicate="intersects")
    hit_idx = sub.index[np.unique(hit[1])]

    affected = final.loc[hit_idx].copy()
    unaffected = final.drop(index=hit_idx)
    affected["_src"] = np.arange(len(affected))

    pairs = gpd.sjoin(affected, mask[["geometry"]], how="inner", predicate="intersects")
    mask_geom = mask.geometry.to_numpy()[pairs["index_right"].to_numpy()]
    inter = shapely.intersection(pairs.geometry.to_numpy(), mask_geom)
    inside = pairs.set_geometry(inter, crs=CRS).drop(columns="index_right")
    inside = inside.explode(index_parts=False)
    inside = inside[(inside.geom_type == "Polygon") & ~inside.geometry.is_empty]
    # Overlapping mask polygons give duplicates – dissolve per source polygon.
    inside = inside.dissolve(by="_src", as_index=False).explode(index_parts=False)
    inside = inside[inside.geom_type == "Polygon"]

    clip = inside.dissolve(by="_src").geometry.reindex(affected["_src"].to_numpy())
    src = affected.geometry.to_numpy()
    has = clip.notna().to_numpy()
    rest = src.copy()
    rest[has] = shapely.difference(src[has], clip[has].to_numpy())
    rest = affected.assign(geometry=rest).explode(index_parts=False)
    rest = rest[(rest.geom_type == "Polygon") & ~rest.geometry.is_empty]

    inside["nin_kode_fkid"] = code
    inside["NiN_name"] = mnk.rules.NIN_NAMES[code]
    inside["nkm_prosess"] = mnk.attributes.NKM_PROSESS[code]
    inside["metode_klassifisering"] = method
    # Directly mapped -> this dataset is the authoritative source for the piece.
    inside["datakilde"] = source

    cols = [c for c in unaffected.columns if not c.startswith("_")]
    out = gpd.GeoDataFrame(
        pd.concat([unaffected[cols], rest[cols], inside[cols]], ignore_index=True), crs=CRS
    )
    print(f"{code}: affected {len(affected):,} -> {len(inside):,} pieces, "
          f"{inside.area.sum() / 1e6:,.1f} km²")
    return out

## 3. Apply direct rules in order

Each rule in `mnk.rules.DIRECT_RULES` is burned in; a later rule overrides an
earlier one where they overlap.

In [4]:
for rule in mnk.rules.DIRECT_RULES:
    mask = load_mask(rule)
    candidates = None
    if rule["where"]:
        candidates = np.logical_and.reduce(
            [final[col].isin(vals) for col, vals in rule["where"].items()]
        )
        candidates = pd.Series(candidates, index=final.index)
    final = override(
        final, mask, rule["nin_kode_fkid"],
        method=rule["metode_klassifisering"],
        source=rule["datakilde"],
        candidates=candidates,
    )
    del mask, candidates
    gc.collect()

print(f"Combined: {len(final):,} polygons")
print(final["nin_kode_fkid"].value_counts())

NA-MC08: 106 polygons (181.5 km²)


NA-MC08: affected 3,046 -> 4,708 pieces, 152.8 km²


NA-MJ01: 3,846 polygons (207.8 km²)


NA-MJ01: affected 7,076 -> 14,660 pieces, 208.5 km²
Combined: 5,119,525 polygons
nin_kode_fkid
NA-MA01    1683947
NA-MA04     552835
NA-MA02     407811
NA-MA05     157864
NA-MA03     154238
NA-MA06      73502
NA-MB02      18545
NA-MC01      17851
NA-MJ01      14660
NA-MC04      13428
NA-MC02       5248
NA-MC08       4708
NA-MF01         14
Name: count, dtype: int64


## 4. Assign nk_fid and save ungridded output

`nk_fid` is assigned per **contiguous same-`nin_kode_fkid` region** (connected
components of adjacent polygons), so every row of one nature-type polygon shares the
same `nk_fid` while keeping its finer attributes (usikkerhet, axes, datakilde,
SeaBee class). Computed by labelling (STRtree adjacency + `scipy` connected
components) — no geometric dissolve.

The result is saved before the grid split as
`marint-naturkart-ungridded_norge_2026_25833.geo.parquet` (EPSG:25833, source for the
PMTiles, `scripts/pyramids.sh`).

In [ ]:
# nk_fid: one id per contiguous same-nin_kode_fkid region (connected component of
# adjacent polygons). Components never cross codes, so each nin_kode_fkid group is
# labelled independently and sequentially to keep memory bounded.
final = final.explode(index_parts=False)
# Drop non-polygon artefacts (difference/intersection can leave boundary lines or
final = final[final.geom_type == "Polygon"].reset_index(drop=True)
geoms = final.geometry.values
code = pd.factorize(final["nin_kode_fkid"])[0]

labels = np.empty(len(final), dtype=np.int64)
next_id = 0
for c in np.unique(code):
    idx = np.flatnonzero(code == c)
    if idx.size == 1:
        labels[idx] = next_id
        next_id += 1
        continue
    sub = geoms[idx]
    p = shapely.STRtree(sub).query(sub, predicate="intersects")
    p = p[:, p[0] < p[1]]
    graph = coo_matrix((np.ones(p.shape[1], dtype=bool), (p[0], p[1])), shape=(idx.size, idx.size))
    ncomp, lab = connected_components(graph, directed=False)
    labels[idx] = lab + next_id
    next_id += ncomp
    del sub, p, graph, lab
    gc.collect()

# number components by Hilbert of their members so nk_fid is spatially coherent
order = pd.Series(final.geometry.hilbert_distance().to_numpy()).groupby(labels).min().sort_values().index
rank = pd.Series(np.arange(1, order.size + 1), index=order)
final["nk_fid"] = ("NKM-" + pd.Series(rank.loc[labels].to_numpy()).astype(str).str.zfill(10)).to_numpy()
print(f"nk_fid: {next_id:,} contiguous polygons from {len(final):,} rows")
del geoms, code, labels, order, rank
gc.collect()

# datamodell usikkerhet-klasse code (lav=1, medium=2, høy=3) so the ungridded is
# fully ready; change it here to re-code without touching 04.
USIKKERHET_KODE = {"lav": 1, "medium": 2, "høy": 3}
final["nkm_usikkerhet"] = final["nkm_usikkerhet"].map(USIKKERHET_KODE).astype("Int64")

ungridded_fname = mnk.utils.to_filename("marint-naturkart-ungridded", "norge", "2026", CRS.split(":")[1])
final.to_parquet(REPO / f"{ungridded_fname}.geo.parquet", compression="snappy")
print(f"Saved {ungridded_fname}.geo.parquet: {len(final):,} polygons")
del final
gc.collect()

## 5. EEA 1 km grid setup

Polygons are split along the EEA 1 km reference grid (`NO_1km`) in its native
**EPSG:3035 (LAEA)**, so cells stay exact 1 km squares and the gridded output is in
3035 (the product is reprojected into it). Each fragment is tagged with `rute_fkid`
(the EEA `CellCode`) and `rute_grense` — `True` if the fragment was cut by a cell
boundary, `False` if the polygon lies entirely within one cell.

In [ ]:
CHUNK = 50_000
GRID_CRS = "EPSG:3035"
UNGRIDDED = REPO / f"{mnk.utils.to_filename('marint-naturkart-ungridded', 'norge', '2026', '25833')}.geo.parquet"

grid = gpd.read_parquet(GRID_URL, columns=["CellCode", "geometry"]).rename(columns={"CellCode": "rute_fkid"})
assert grid.crs.to_epsg() == 3035

_bb = json.loads(pq.read_schema(UNGRIDDED).metadata[b"geo"])["columns"]["geometry"]["bbox"]
minx, miny, maxx, maxy = gpd.GeoSeries([shapely.box(*_bb)], crs=CRS).to_crs(GRID_CRS).total_bounds
minx, miny, maxx, maxy = minx - 5000, miny - 5000, maxx + 5000, maxy + 5000
GRID_BOUNDS = (minx, miny, maxx, maxy)
grid = grid.cx[minx:maxx, miny:maxy].reset_index(drop=True)
grid_geom = grid.geometry.to_numpy()
grid_id = grid["rute_fkid"].to_numpy()
grid_sindex = grid.sindex
print(f"Grid cells in extent: {len(grid):,}")


def split_chunk(chunk):
    """Split *chunk* by the grid; fragments carry rute_fkid and rute_grense."""
    src_idx, cell_idx = grid_sindex.query(chunk.geometry, predicate="intersects")
    geoms = chunk.geometry.to_numpy()[src_idx]
    cells = grid_geom[cell_idx]
    # Polygons fully inside one cell are not cut; the rest get a grid-cell edge.
    inside = shapely.contains_properly(cells, geoms) | shapely.covers(cells, geoms)
    out_geom = geoms.copy()
    out_geom[~inside] = shapely.intersection(geoms[~inside], cells[~inside])
    out = chunk.iloc[src_idx].drop(columns="geometry").reset_index(drop=True)
    out["rute_fkid"] = grid_id[cell_idx]
    out["rute_grense"] = ~inside
    out = gpd.GeoDataFrame(out, geometry=out_geom, crs=GRID_CRS)
    out = out.explode(index_parts=False)
    out = out[(out.geom_type == "Polygon") & ~out.geometry.is_empty]
    return out.reset_index(drop=True)

Grid cells in extent: 1,026,232


## 6. Split, assign fragment ids, sort, and save

Each polygon is split by the grid; every fragment gets a unique `nk_fragment_id`
(`<nk_fid>-NNNN`) while keeping the shared `nk_fid` from §4, so fragments stay linked
to their parent polygon. The output is sorted by `rute_fkid`, then Hilbert distance
within each cell, for downstream performance.

**Memory**: the split runs in chunks to parquet parts; the parts are concatenated and
sorted at the pyarrow level (compact WKB geometry, not shapely objects), and the
GeoParquet is written in sorted slices.

In [ ]:
fname = mnk.utils.to_filename("marint-naturkart", "norge", "2026", GRID_CRS.split(":")[1])
parquet_path = REPO / f"{fname}.geo.parquet"

OUT_COLS = [
    "nk_fid", "nk_fragment_id", "opphav_kode", "rute_fkid", "rute_grense",
    "nin_kode_fkid", "NiN_name", "nkm_prosess",
    mnk.rules.COL_LYS, mnk.rules.COL_SALINITET, mnk.rules.COL_SUBSTRAT, mnk.rules.COL_TOERFALL,
    mnk.rules.COL_SA_FA,
    "metode_klassifisering", "nkm_usikkerhet", "metode_usikkerhet", "datakilde", "nkm_seabee_klasse",
]

pf_in = pq.ParquetFile(UNGRIDDED)
with tempfile.TemporaryDirectory(dir=REPO) as tmp:
    # Phase 1: stream the ungridded parquet (no in-memory `final`), reproject each

    parts = []
    done = 0
    for bi, batch in enumerate(pf_in.iter_batches(batch_size=CHUNK)):
        tb = pa.Table.from_batches([batch])
        gm = shapely.from_wkb(tb.column("geometry").to_pandas().to_numpy())
        df = tb.drop_columns(["geometry"]).to_pandas()
        chunk = gpd.GeoDataFrame(df, geometry=gm, crs=CRS).to_crs(GRID_CRS)
        chunk["geometry"] = shapely.make_valid(chunk.geometry.to_numpy())
        out = split_chunk(chunk)
        out["_hilbert"] = out.geometry.hilbert_distance(total_bounds=GRID_BOUNDS)
        part = Path(tmp) / f"part_{bi:05d}.parquet"
        out.to_parquet(part, compression="snappy")
        parts.append(part)
        done += len(chunk)
        print(f"  split {done:,}/{pf_in.metadata.num_rows:,} -> {len(out):,}")
        del tb, gm, df, chunk, out
        gc.collect()
    del grid, grid_geom, grid_id, grid_sindex
    gc.collect()

    geo = json.loads(pq.read_schema(parts[0]).metadata[b"geo"])
    for col in geo["columns"].values():
        col.pop("bbox", None)

    # Global order by (rute_fkid, Hilbert). Rows are written in sorted slices so no
    # full sorted copy of the table is held (peak ~ one table, not 2x).
    table = ds.dataset([str(p) for p in parts], format="parquet").to_table()
    # to_table() concatenates the part chunks; the combined WKB column exceeds the
    # 32-bit binary offset limit, so widen it to large_binary for take() 
    gi = table.schema.get_field_index("geometry")
    table = table.set_column(gi, "geometry", pc.cast(table.column("geometry"), pa.large_binary()))
    idx = pc.sort_indices(table, sort_keys=[("rute_fkid", "ascending"), ("_hilbert", "ascending")])

    nk_fid = table.column("nk_fid").take(idx).to_pandas()
    frag = (nk_fid.groupby(nk_fid).cumcount() + 1).map("{:04d}".format)
    nk_fragment_id = nk_fid.str.cat(frag, sep="-").to_numpy()
    del nk_fid, frag

    schema = (
        table.slice(0, 0)
        .append_column("nk_fragment_id", pa.array([], type=pa.string()))
        .select(OUT_COLS + ["geometry"])
        .schema.with_metadata({b"geo": json.dumps(geo).encode()})
    )
    schema = schema.set(schema.get_field_index("geometry"), pa.field("geometry", pa.binary()))
    nrows = table.num_rows
    with pq.ParquetWriter(parquet_path, schema, compression="snappy") as w:
        for i in range(0, nrows, 500_000):
            sl = table.take(idx[i : i + 500_000])
            sl = sl.append_column("nk_fragment_id", pa.array(nk_fragment_id[i : i + 500_000], type=pa.string()))
            w.write_table(sl.select(OUT_COLS + ["geometry"]).cast(schema))
            print(f"  write {min(i + 500_000, nrows):,}/{nrows:,}")
            del sl
    del table, idx, nk_fragment_id
    gc.collect()

print(f"Saved {nrows:,} fragments -> {parquet_path.name}")